In [23]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openrouter import ChatOpenRouter
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from operator import add

In [ ]:
load_dotenv()
llm_model = ChatOpenRouter(model="meta-llama/llama-3-8b-instruct")
# llm_model = ChatOpenRouter(model="openai/gpt-oss-120b:free")

In [25]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback about the provided input")
    score: int   = Field(description="Score for the given input, out of 10", ge=0, le=10)


In [ ]:
st_model = llm_model.with_structured_output(EvaluationSchema)

In [27]:
class EssayState(TypedDict):
    essay: str
    cr_feedback: str # content / relevance
    os_feedback: str # organisation / structure
    lg_feedback: str # language / grammar
    final_feedback: str
    individual_score: Annotated[list[int], add]
    final_score: float

In [ ]:
def evaluate_cr(state: EssayState):
    essay = state['essay']

    prompt = f'''
You're an experienced essay evaluator. You are provided with an essay. Your have to evaluate the content and relevance of this essay.

# Points to ponder:
- Does the essay stay on the topic of AI?
- Are the ideas meaningful, accurate, and relevant?
- Does it cover useful points like definition, uses, benefits, concerns?

# Essay: {essay}
'''
    
    evaluation = st_model.invoke(prompt)

    return {'cr_feedback': evaluation.feedback, 'individual_score': [evaluation.score]}

In [29]:
def evaluate_os(state: EssayState):
    essay = state['essay']

    prompt = f'''
You're an experienced essay evaluator. You are provided with an essay. Your have to evaluate the organization and structure of this essay.

# Points to ponder:
- Does it have a clear beginning, middle, and conclusion?
- Are sentences logically connected?
- Is the flow smooth and easy to follow?

# Essay: {essay}
'''
    
    evaluation = st_model.invoke(prompt)

    return {'os_feedback': evaluation.feedback, 'individual_score': [evaluation.score]}

In [30]:
def evaluate_lg(state: EssayState):
    essay = state['essay']

    prompt = f'''
You're an experienced essay evaluator. You are provided with an essay. Your have to evaluate the language and grammar of this essay.

# Points to ponder:
- Correct grammar, spelling, punctuation
- Proper vocabulary usage
- Clear and readable sentence formation

# Essay: {essay}
'''
    
    evaluation = st_model.invoke(prompt)

    return {'lg_feedback': evaluation.feedback, 'individual_score': [evaluation.score]}

In [34]:
def final_evaluation(state: EssayState):
    cr_feedback = state['cr_feedback']
    os_feedback = state['os_feedback']
    lg_feedback = state['lg_feedback']
    individual_score = state['individual_score']

    prompt = f'''
You're an experienced essay evaluator. You're provided with the following details about an essay:

# Factors of evaluation:
- Content and relevance feedback: {cr_feedback}
- Organization and structure feedback: {os_feedback}
- Language and grammar feedback: {lg_feedback}

Your task is to give a final feedback on the basis of above feedbacks.
'''
    
    evaluation = llm_model.invoke(prompt)

    final_feedback = evaluation.content
    final_score = sum(individual_score)/3

    return {'final_feedback': final_feedback, 'final_score': final_score}

In [32]:
graph = StateGraph(EssayState)

graph.add_node('evaluate_cr', evaluate_cr)
graph.add_node('evaluate_os', evaluate_os)
graph.add_node('evaluate_lg', evaluate_lg)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_cr')
graph.add_edge(START, 'evaluate_os')
graph.add_edge(START, 'evaluate_lg')
graph.add_edge('evaluate_cr', 'final_evaluation')
graph.add_edge('evaluate_os', 'final_evaluation')
graph.add_edge('evaluate_lg', 'final_evaluation')
graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [33]:
essay = '''Artificial Intelligence (AI) is one of the most important technologies of the modern world. It enables machines to perform tasks that usually require human intelligence, such as learning, reasoning, problem-solving, and decision-making. AI is used in many fields, including healthcare, education, banking, transportation, and entertainment. Virtual assistants, recommendation systems, and self-driving cars are common examples of AI applications. It helps improve efficiency, accuracy, and productivity in daily life and business operations. However, AI also raises concerns about privacy, jobs, and ethics. With proper use and responsible development, AI can greatly benefit society in the future.'''

input_state = {'essay': essay}
output_state = workflow.invoke(input_state)

print(output_state['final_feedback'])
print(output_state['final_score'])

---------------- 1 ----------------
---------------- 2 ----------------
---------------- 3 ----------------
---------------- 4 ----------------
evaluation ->  content='**Overall Assessment (4\u202f/\u202f10)**  \n\nYour essay demonstrates a solid grasp of basic English mechanics and stays on the assigned topic of artificial intelligence. However, the piece falls short in two critical areas: depth of content and structural coherence. Below is a concise breakdown of the main strengths you can build on and the key areas that need improvement.\n\n---\n\n### 1. Content & Relevance  \n**What works:**  \n- You correctly define AI in a single sentence.  \n- You mention a variety of domains (healthcare, finance, transportation, etc.) and note generic benefits such as “efficiency” and “productivity.”  \n- You briefly acknowledge common concerns (privacy, job displacement, ethics).\n\n**Where it falls short:**  \n- **Superficial coverage:**\u202fThe essay lists topics without explaining *how* AI 